In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def equiv(lo, hi, margin):
    """**등가성 판정**(R29 ①). 미결을 등가로 읽지 않기 위해 별도 함수로 둔다.
    CI **전체**가 ±margin 안에 들어가야 「등가」다."""
    if lo > -margin and hi < margin:
        return "✅ 등가"
    if lo > margin or hi < -margin:
        return "❌ 차이 있음"
    return "⚠️ 미결"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, glob, importlib, time
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0   = 20260803
IDX_S   = 1
NB_BOOT = 4000

# ── 창 (R = index 100 · 360Hz)
#    ★ **`stt_32` 신설** — Q7-M 은 `P_late`(폭 32)를 `STT`(폭 85)와 비교했다.
#      창이 넓으면 기계적으로 정보가 더 들어간다(R27 ③). **폭을 맞춘 음성 대조**가 필요하다.
SEGS = {"p_full": (0, 85), "p_early": (0, 32), "p_late": (53, 85),
        "stt": (130, 215), "stt_32": (130, 162), "stt_safe": (130, 175)}
WIDTH_PAIRS = (("p_late", "stt_32"), ("p_full", "stt"))     # (폭 32, 32) · (폭 85, 85)

# ── 사전등록 상수. **이 아래 어느 셀에서도 다시 고르지 않는다.**
# ★ 기저 밀도는 **픽스처 실측**으로 정했다 — (4,8,16,32) 로는 기저 밖 `f2_12` 가
#   0.5206 으로 새고(여유 0.02 초과), 아래처럼 촘촘히 하면 최대 이탈 **0.0140** 로 든다.
BASIS_K   = (4, 6, 8, 12, 16, 24, 32)   # 잔차화 기저의 국소 기저선 창
PROBE_K   = (5, 10, 20)                 # ★ **기저에서 빼둔** 프로브 — N2a 가 이걸로 검증
HIST_K    = 64                  # N4 확장 기저
LB_K      = 16                  # 보고용 대표 (Q7-M 승계)
MIN_S_TPL = 20
K_FOLD    = 5                   # R22
N_REPEAT  = 3
N_SHUF    = 20                  # R26 ② — null 의 셔플 오차를 CI 에 전파
TREND_W   = 8                   # N4 국소 추세 창

PRIMARY_BASIS = "lin"           # ★ 주 기저(사전등록). rank 는 **민감도·삼각검증**
PROBE_MARGIN = 0.02             # ★ N2 전제 — 프로브의 잔차 AUROC 가 0.5±이 안이어야
EQUIV_MARGIN = 0.05             # ★ N3 등가 여유. Q7-M CI 반폭 0.0684(n=18) →
                                #   59개체면 반폭 ≈0.038 이라 **도달 가능**하다(사전 계산)
BONF3     = 0.05 / 3 / 2        # 1차 가족 {N3a · N3b · N4}
ISO_HI, ISO_LO = 0.7, 0.3

# ★★ 사전등록 규칙 체크리스트 (R29 ③) — 규칙을 아는 것과 적용하는 것은 다른 일이다.
#    Q7-M 은 R27 ③ 을 쓰고 바로 다음 사전등록에서 어겼다. 그래서 명시적으로 박는다.
RULE_CHECK = {
    "R16 fallback 없음":          "자산 셀에서 예외 삼킴 없음 · 목록 실패 시 중단",
    "R22 교차적합":               "겹 밖 점수 · 반복 CV",
    "R24-b ② 무너져야 할 대조군": "프로브(f2_6·12·24·f1_rank)를 함께 채점",
    "R25 max 바닥 없음":          "어떤 바닥도 개체별 max 로 만들지 않는다",
    "R26 ② 자기 null":            "학습 낀 팔은 라벨셔플 null 위 초과분 · SE 전파",
    "R27 ② 차이 우선":            "수준보다 (P 초과 − 음성대조 초과) 를 1차로",
    "R27 ③ 폭 정합":              "★ p_late(32) vs stt_32(32) · p_full(85) vs stt(85)",
    "R28 ① 파라미터화":           "★ k 를 하나가 아니라 4~32 촘촘히 · 밀도는 픽스처 실측",
    "R29 ④ 함수형도 검증":        "★ f1_rank 프로브 + rank 기저 민감도",
    "R28 ② 하류 변수 금지":       "★ post_rr 은 기저에 **넣지 않는다**",
    "R28 ③ 물려받은 상수 감사":   "MIN_CELL 계열 없음(잔차화는 칸을 안 쓴다)",
    "R29 ① 등가는 여유 사전등록": "★ ±0.05 · 필요 표본 미리 계산(59개체 반폭 ≈0.038)",
    "R29 ② 측정 불가 분기 금지":  "★ ⛔ 판정은 어떤 결론 분기도 타지 않는다",
    "R29 ④ 강건성도 CI":          "삼각검증·층별을 CI 와 함께",
}

CONFIG = dict(
    exp="quest46_q7n_residualize", quest="ailab-2026-0046", step="svdb-residualize",
    parent_exp=["quest46_q7m_recover_power", "ailab-2026-0062"],
    purpose=("정합은 목표를 달성했고 한계에 닿았다 — Q7-M 이 `f1`·`f2_16` 을 동시에 "
             "통제했지만(M1·M2 ✅) 남은 미결 셋이 전부 **정합이 표본을 버린다**에서 "
             "온다: ① 3축 = 6/59(Q7-K·Q7-M 둘 다) ② `k` 는 무한 계열이라 `f2_8` 이 "
             "8.1σ 로 생존 ③ 등가성엔 개체 34 가 필요한데 18. **잔차화는 축을 몇 개든 "
             "한꺼번에 넣으면서 개체 59 · S 100% 를 쓴다.** 정합 대신 점수에서 조기성 "
             "성분을 회귀로 제거하고, **기저에서 빼둔 프로브**로 통제를 검증한다"),
    dataset="SVDB 전수 · svdb_data5.npz + Q7-B 예측 캐시(라벨·매핑용)",
    basis_k=list(BASIS_K), probe_k=list(PROBE_K), hist_k=HIST_K, n_shuffle=N_SHUF,
    windows={k: list(v) for k, v in SEGS.items()},
    width_pairs=[list(p) for p in WIDTH_PAIRS],
    margins=dict(probe=PROBE_MARGIN, equiv=EQUIV_MARGIN),
    rule_check=RULE_CHECK,
    predictions={
        "N1": "(관문 아님) 잔차화 전/후 각 팔 · 기저 3종(선형 · 이차 · 이력확장)",
        "N2a": f"★ **전제 · k 커버리지** — 기저에서 빼둔 `f2_k`(k={PROBE_K}) 프로브의 "
               f"잔차 AUROC 가 **전부** 0.5±{PROBE_MARGIN}. 기저 **안** 변수의 잔차는 "
               "정의상 0 이라 검증이 안 된다 — **빼둔 것으로 검증해야** 「`k` 계열을 "
               "덮었다」를 말할 수 있다. 미충족이면 아래 **수준**을 봉인한다",
        "N2b": f"**함수형** — 단조 변환 프로브 `f1_rank` 가 0.5±{PROBE_MARGIN}. "
               "★ **선형 기저로는 실패가 예상된다**(픽스처 실측 0.5432 · drift 0 에서 "
               "0.3555). 순위 기저에서는 `f1_rank` 가 기저 안이라 **정의상 0.5** 다 — "
               "그래서 **`rank` 기저를 민감도로 병기**하고, 두 기저에서 N3 결론이 같은지 "
               "본다(함수형 선택에 대한 삼각검증). **N2b 실패는 봉인 사유가 아니라 "
               "캐비앳**이다 — 차이(N3)는 두 팔을 같게 오염시키므로 살아남는다(R27 ②)",
        "N3a": f"★ **등가**(주 기저 lin) 폭 32 — (p_late 초과) − (stt_32 초과) CI ⊂ ±{EQUIV_MARGIN} "
               "(Bonferroni 3). 등가면 **처음으로 「P 창은 음성 대조와 구별되지 "
               "않는다」를 확정**할 수 있다(R29 ①)",
        "N3b": f"★ **등가** 폭 85 — (p_full 초과) − (stt 초과) CI ⊂ ±{EQUIV_MARGIN}",
        "N4": "★ **이력 확장 기저**(+f4 +국소추세 +f2_64) 잔차에서 STT 초과 CI 하한 > 0 "
              "(Bonferroni 3). rate hysteresis 는 **선행 이력**의 문제이므로 이렇게 "
              "묻는다 — ⛔ `post_rr` 은 하류 변수라 넣지 않는다(R28 ②)",
        "N5": "(관문 아님) 층별(고립/혼합/런) — 잔차화는 59개체를 다 쓰므로 층이 산다",
        "N6": "(관문 아님) 삼각검증 — 층화 판본(f1 × f2_16 칸)이 같은 부호인가"},
    caveat=("★ **잔차화의 한계를 미리 적는다**: ① **선형 성분만** 제거한다 → 이차항 "
            "기저를 민감도로 병기한다 ② 조기성에 귀속되는 형태 성분까지 지우므로 "
            "**보수적**이다(형태를 과소평가하는 방향) ③ 잔차화는 **라벨을 안 쓰므로** "
            "누수가 없고, 순수 특징의 영분포는 그대로 0.5 다 — 학습이 낀 팔만 셔플 null "
            "이 필요하다. ★ **N2a 가 전제**다 — `k` 프로브가 안 죽으면 「`k` 계열을 "
            "덮었다」고 말할 수 없고, 그러면 아래 **수준**을 인용하지 않는다. "
            "**N2b(함수형)는 봉인 사유가 아니라 캐비앳**이다 — 실패하면 「단조 비선형 "
            "성분은 안 지워졌다」를 명시하고 `rank` 기저 결과를 나란히 낸다. 단 **차이**(N3a·N3b)와 "
            "반응 곡선은 공통 오염이 상쇄되므로 봉인 아래에서도 읽는다(R27 ②). "
            "★ **층화 삼각검증은 「비트 100%」가 아니다** — 칸 안에 두 클래스가 있어야 "
            "한다. 이득은 **개체 수준 문턱이 없다**는 것이다. 그렇게 정확히 적는다. "
            "★ 개체 내부 템플릿·로지스틱은 **라벨을 쓰는 상한**이지 배포 모형이 아니다. "
            "학습 0회 · GPU 불필요 · 예상 20~35분(셔플 20회 × 59개체가 지배)"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7n_residualize", CONFIG, project=PROJECT)
run.log("설정 ✅ 잔차화 · 기저 k=" + str(BASIS_K) + " · 프로브 k=" + str(PROBE_K))
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [{'x'}] {k_:<26} {v_}")

In [ ]:
# CELL 2 — 【N-0a】 자산 · 매핑 (Q7-D~M 동일 규약 — fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다 — 연속 번호로 대체하지 않는다")
labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels) and not (set(REC.tolist()) - set(recs))

d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"길이 불일치 {int(keep.sum())} vs {len(Y)}"
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
POST = np.asarray(d5["post_rr"])[keep].astype(float)
BEAT = np.asarray(d5["beat"])[keep]
WIN = {k: np.ascontiguousarray(BEAT[:, :, a:b]).astype("float32")
       for k, (a, b) in SEGS.items()}
del BEAT
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【N-0a】 자산 · 매핑")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(ALLR)}개")
run.log("  창 폭 — " + " · ".join(f"{k}:{v[1]-v[0]}" for k, v in SEGS.items()))
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【N-A】 특징 · 기저 · 프로브 · 점수
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

def local_base(pre_v, k):
    n = len(pre_v); out = np.empty(n); med = float(np.median(pre_v))
    for i in range(n):
        a = max(0, i - k)
        out[i] = med if i - a < 3 else float(np.median(pre_v[a:i]))
    return out

def trend(pre_v, w):
    """직전 `w` 박동 RR 의 기울기 — 이력(rate hysteresis)의 대리. **선행 정보만** 쓴다."""
    n = len(pre_v); out = np.zeros(n); x = np.arange(w, dtype=float)
    xc = x - x.mean(); den = float((xc * xc).sum())
    for i in range(n):
        a = i - w
        if a < 0:
            continue
        y = pre_v[a:i]
        out[i] = float(((y - y.mean()) * xc).sum() / den)
    return out

def all_feats(pre_v, post_v):
    """기저 · 프로브 · 보고용 특징을 한 번에. **`post_rr` 은 기저에 안 들어간다**(R28 ②)."""
    med = float(np.median(pre_v)); n = len(pre_v)
    F = {}
    F["f1"] = med - pre_v
    for k in sorted(set(BASIS_K) | set(PROBE_K) | {HIST_K}):
        F[f"f2_{k}"] = 1.0 - pre_v / np.maximum(local_base(pre_v, k), 1e-9)
    b16 = local_base(pre_v, LB_K)
    b_first = local_base(pre_v, BASIS_K[0])
    F["f6"] = 1.0 - b16 / max(med, 1e-9)
    cv = np.empty(n)
    for i in range(n):
        a = max(0, i - TREND_W); w_ = pre_v[a:i] if i - a >= 3 else pre_v[:3]
        cv[i] = float(np.std(w_) / max(np.mean(w_), 1e-9))
    F["f4"] = cv
    F["trend"] = trend(pre_v, TREND_W)
    F["f5"] = np.r_[0.0, F[f"f2_{BASIS_K[0]}"][:-1]]
    # ★ 단조 비선형 프로브 — 선형 기저가 `f1` 의 단조 변환까지 덮는지 본다
    F["f1_rank"] = stats.rankdata(F["f1"]) / n
    # 보고용(기저 아님)
    F["f3"] = 1.0 - (pre_v + post_v) / np.maximum(2.0 * b_first, 1e-9)
    F["f3_glob"] = 1.0 - (pre_v + post_v) / (2.0 * max(med, 1e-9))
    return F

def basis_mat(F, kind):
    """기저 행렬. ★ **열을 z-표준화**한다 — `f1` 은 샘플 단위(±100)이고 `f2_k`·`f6` 은
    무차원(±0.5)이라 스케일이 100배 넘게 벌어진다. 그대로 `lstsq` 에 넣으면 조건수가
    나빠 **필요한 특이방향이 잘려** 기저 안 변수조차 완전히 안 빠진다(픽스처가 잡았다).
    열 스케일링은 열공간을 안 바꾸므로 잔차는 수학적으로 동일하다."""
    cols = ["f1"] + [f"f2_{k}" for k in BASIS_K] + ["f6"]
    Z = np.stack([F[c] for c in cols], axis=1)
    if kind == "quad":
        Z = np.c_[Z, Z ** 2]
    elif kind == "hist":
        Z = np.c_[Z, F["f4"], F["trend"], F[f"f2_{HIST_K}"]]
    elif kind not in ("lin", "rank"):
        raise AssetError(f"기저 종류 {kind} 를 모른다")
    elif kind == "rank":
        # ★ AUROC 는 **순위 통계량**이다. 순위 공간에서 잔차화하면 조기성의 **단조
        #   비선형** 성분까지 정의상 흡수한다 — 선형 기저가 못 지우는 부분이다.
        Z = np.stack([stats.rankdata(F[c]) / len(F[c]) for c in cols], axis=1)
    Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-12)
    return np.c_[np.ones(len(Z)), Z]

def prep(s, kind):
    """`rank` 기저에서는 **점수도 순위로** 바꿔 같은 공간에서 잔차화한다."""
    s = np.asarray(s, float)
    return stats.rankdata(s) / len(s) if kind == "rank" else s

RESID_EPS = 1e-9        # 잔차가 이 배율 아래면 **수치적으로 0** 으로 본다

def residualize(s, Z):
    """점수에서 기저의 **선형 성분**을 뺀다. **라벨을 안 쓰므로 누수가 없다.**

    ★ 기저 **안**에 있는 변수는 잔차가 정의상 0 인데, 부동소수점에서는 ~1e-13 의
    잡음이 남고 그 잡음의 AUROC 는 **0.5 근처의 난수**가 된다(픽스처에서 0.4759).
    그걸 값으로 보고하면 오독을 부르므로 **정확히 0 으로 접는다** → AUROC 0.5."""
    s = np.asarray(s, float)
    if not np.isfinite(s).all():
        return None
    beta, *_ = np.linalg.lstsq(Z, s, rcond=None)
    e = s - Z @ beta
    if float(np.std(e)) <= RESID_EPS * (float(np.std(s)) + 1e-12):
        return np.zeros_like(e)
    return e

def dist(B, ref):
    d = B - ref[None]
    return np.sqrt((d * d).sum(axis=(1, 2)))

def cv_logit(X, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            mu = X[tr].mean(0); sd = X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit((X[tr] - mu) / sd, tt[tr].astype(int))
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def two_template_cv(B, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            medN = np.median(B[tr & ~tt], axis=0); medS = np.median(B[tr & tt], axis=0)
            sc[te] = dist(B[te], medN) - dist(B[te], medS)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def run_struct(t_):
    prev_ = np.r_[False, t_[:-1]]; next_ = np.r_[t_[1:], False]
    iso = t_ & ~prev_ & ~next_
    rl = mx = 0
    for v in t_:
        rl = rl + 1 if v else 0
        mx = max(mx, rl)
    return float(iso.sum() / max(t_.sum(), 1)), int(mx)

LR_COLS = ["f1", "f2_8", "f2_16", "f3", "f4", "f5", "f6"]

def build_scores(F, tt, MORPH, seed):
    """라벨을 쓰는 팔만 여기서 만든다(템플릿 · 로지스틱). **셔플 null 도 같은 함수.**"""
    S = {}
    X = np.stack([F[c] for c in LR_COLS], axis=1)
    arms = {"lr_all": X, "lr_norr": X[:, [LR_COLS.index("f3"), LR_COLS.index("f4")]],
            "lr_f1": X[:, [LR_COLS.index("f1")]]}
    for nm_, XX in arms.items():
        sc_ = cv_logit(XX, tt, K_FOLD, seed, N_REPEAT)
        if sc_ is None:
            return None
        S[nm_] = sc_
    for nm_, B_ in MORPH.items():
        st = two_template_cv(B_, tt, K_FOLD, seed, N_REPEAT)
        S[nm_] = st if st is not None else np.full(len(tt), np.nan)
    return S

run.log("\n" + "=" * 100)
run.log("【N-A】 특징 · 기저 · 점수 계산")
run.log("=" * 100)
T0 = time.time()
FEAT, SCORES, META, SKIP = {}, {}, {}, []
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    if int(tt.sum()) < MIN_S_TPL or int((~tt).sum()) < MIN_S_TPL:
        SKIP.append((int(r), f"S {int(tt.sum())} · N {int((~tt).sum())}")); continue
    F = all_feats(PRE[mm], POST[mm])
    MORPH = {k: WIN[k][mm] for k in SEGS}
    S = build_scores(F, tt, MORPH, SEED0)
    if S is None:
        SKIP.append((int(r), "겹 안 클래스 부족")); continue
    iso_f, mx_run = run_struct(tt)
    FEAT[int(r)] = F; SCORES[int(r)] = S
    META[int(r)] = dict(tt=tt, MORPH=MORPH, pos=int(tt.sum()), prev=float(tt.mean()),
                        iso_frac=iso_f, max_run=mx_run, n=int(len(mm)))
RS = sorted(SCORES)
run.log(f"  채점 **{len(RS)}개체** · 제외 {len(SKIP)}개체 · {time.time()-T0:.0f}초")
run.log("  ▸ 정합과 달리 **개체를 안 버린다** — Q7-M 은 18/59 였다")
PURE = ["f1", "f2_16", "f3", "f3_glob", "f4", "f5", "f6"]
PROBES = [f"f2_{k}" for k in PROBE_K] + ["f1_rank"]
MORPH_ARMS = list(SEGS)
LR_ARMS = ["lr_all", "lr_norr", "lr_f1"]
ARMS = PURE + PROBES + MORPH_ARMS + LR_ARMS
LABEL_ARMS = MORPH_ARMS + LR_ARMS          # 셔플 null 이 필요한 팔

def score_of(r, a):
    return SCORES[r][a] if a in SCORES[r] else FEAT[r][a]

RAW = {a: np.array([roc_auc_score(META[r]["tt"].astype(int), score_of(r, a))
                    if np.isfinite(score_of(r, a)).all() else np.nan for r in RS])
       for a in ARMS}
run.log("  무잔차 매크로 — " + " · ".join(f"{a} {np.nanmean(RAW[a]):.4f}" for a in
        ("f1", "f2_16", "f3", "stt", "stt_32", "p_late", "p_full", "lr_all")))
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【N-B】 ★ 잔차화 — 기저 3종 (선형 · 이차 · 이력확장)
run.log("\n" + "=" * 100)
run.log("【N-B】 잔차화 — 점수에서 조기성의 선형 성분을 뺀다")
run.log("=" * 100)
BASES = ("lin", "rank", "quad", "hist")
ZMAT = {r: {k: basis_mat(FEAT[r], k) for k in BASES} for r in RS}
RES = {k: {a: np.full(len(RS), np.nan) for a in ARMS} for k in BASES}
for k in BASES:
    for i, r in enumerate(RS):
        tt = META[r]["tt"]; Z = ZMAT[r][k]
        for a in ARMS:
            e = residualize(prep(score_of(r, a), k), Z)
            if e is None:
                continue
            RES[k][a][i] = roc_auc_score(tt.astype(int), e)
run.log("  기저 차원 — " + " · ".join(f"{k} {ZMAT[RS[0]][k].shape[1]}" for k in BASES))
run.log("\n  팔별 매크로 — 무잔차 → **lin** · rank · quad · hist")
for a in ARMS:
    star = "  ★" if a in PROBES else ("  ▸" if a in MORPH_ARMS else "")
    run.log(f"    {a:<9} {np.nanmean(RAW[a]):.4f} → **{np.nanmean(RES['lin'][a]):.4f}** · "
            + " · ".join(f"{np.nanmean(RES[k][a]):.4f}" for k in BASES[1:]) + star)
run.log("\n  ▸ 기저 **안**에 있는 변수(f1·f2_16·f6)의 잔차는 **정의상 0** 이라 0.5 다 —")
run.log("    통제 검증이 안 된다. **기저에서 빼둔 프로브**(★)가 죽어야 「k 계열을 덮었다」")
CONFIG["residual"] = {k: {a: float(np.nanmean(RES[k][a])) for a in ARMS} for k in BASES}
CONFIG["raw"] = {a: float(np.nanmean(RAW[a])) for a in ARMS}
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【N-C】 라벨셔플 null (기저 3종 동시 · SE 를 CI 에 전파 · R26 ②)
# 잔차화는 **라벨을 안 쓰므로** 순수 특징의 영분포는 그대로 0.5 다. 학습이 낀 팔
# (템플릿 · 로지스틱)만 셔플이 필요하다 — Q7-M 의 `lr_f1` 0.4216 이 그 증거였다.
run.log("\n" + "=" * 100)
run.log(f"【N-C】 라벨셔플 null (셔플 {N_SHUF}회 · {len(RS)}개체 · 학습 낀 팔만)")
run.log("=" * 100)
T1 = time.time()
NULL = {k: {a: np.full(len(RS), np.nan) for a in ARMS} for k in BASES}
NSE  = {k: {a: np.full(len(RS), np.nan) for a in ARMS} for k in BASES}
for k in BASES:
    for a in ARMS:
        if a not in LABEL_ARMS:
            NULL[k][a][:] = 0.5; NSE[k][a][:] = 0.0
for i, r in enumerate(RS):
    tt = META[r]["tt"]
    acc = {k: {a: [] for a in LABEL_ARMS} for k in BASES}
    for s_ in range(N_SHUF):
        rng = np.random.RandomState(SEED0 + 7919 * (s_ + 1) + int(r))
        ts = rng.permutation(tt)                       # 유병률 보존
        Ss = build_scores(FEAT[r], ts, META[r]["MORPH"], SEED0 + 31 * (s_ + 1))
        if Ss is None:
            continue
        for k in BASES:
            Z = ZMAT[r][k]
            for a in LABEL_ARMS:
                e = residualize(prep(Ss[a], k), Z)
                if e is None:
                    continue
                acc[k][a].append(roc_auc_score(ts.astype(int), e))
    for k in BASES:
        for a in LABEL_ARMS:
            v_ = np.asarray(acc[k][a], float)
            if len(v_) >= 2:
                NULL[k][a][i] = float(v_.mean())
                NSE[k][a][i] = float(v_.std(ddof=1) / np.sqrt(len(v_)))
run.log(f"  ({time.time()-T1:.0f}초)  **선형 기저** — 실측 vs null(±SE) vs 초과분")
for a in ARMS:
    m_ = np.nanmean(RES["lin"][a]); n_ = np.nanmean(NULL["lin"][a])
    se_ = np.nanmean(NSE["lin"][a])
    star = "  ★" if a in PROBES else ("  ▸" if a in MORPH_ARMS else "")
    run.log(f"    {a:<9} {m_:.4f}  null {n_:.4f} ±{se_:.4f}  **초과 {m_-n_:+.4f}**{star}")
CONFIG["null"] = {k: {a: float(np.nanmean(NULL[k][a])) for a in ARMS} for k in BASES}
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【N-D】 관문 N2(전제) · N3a · N3b(등가) · N4
def boot_excess(a, base, seed, nb=NB_BOOT, q=2.5, const=None):
    """초과분(또는 두 팔 초과분의 차이)의 개체 부트스트랩 + **null 오차 전파**(R26 ②)."""
    d1 = RES[base][a] - NULL[base][a]; s1 = np.nan_to_num(NSE[base][a])
    if const is not None:
        d, se = RES[base][a] - const, np.zeros(len(RS))
    else:
        d, se = d1, s1
    ok = np.isfinite(d); d = d[ok]; se = se[ok]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.empty(nb)
    for b in range(nb):
        ix = rng.randint(0, len(d), len(d))
        v[b] = (d[ix] - rng.normal(0.0, 1.0, len(ix)) * se[ix]).mean()
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

def boot_pair(a1, a2, base, seed, nb=NB_BOOT, q=2.5):
    d = (RES[base][a1] - NULL[base][a1]) - (RES[base][a2] - NULL[base][a2])
    se = np.sqrt(np.nan_to_num(NSE[base][a1]) ** 2 + np.nan_to_num(NSE[base][a2]) ** 2)
    ok = np.isfinite(d); d = d[ok]; se = se[ok]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.empty(nb)
    for b in range(nb):
        ix = rng.randint(0, len(d), len(d))
        v[b] = (d[ix] - rng.normal(0.0, 1.0, len(ix)) * se[ix]).mean()
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

run.log("\n" + "=" * 100)
run.log(f"【N-D】 관문 (선형 기저 · {len(RS)}개체)")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<5}{v}  {d}")

# ── N2a ★ 전제(k 커버리지) · N2b 함수형 — **둘을 분리한다**
#    기저 안 변수의 잔차는 정의상 0 이라 검증이 안 된다. 빼둔 것으로만 검증 가능하다.
KPROBE = [f"f2_{k}" for k in PROBE_K]
for bs in ("lin", "rank"):
    run.log(f"  N2 프로브 [{bs} 기저] (여유 ±{PROBE_MARGIN})")
    okk = []
    for a in KPROBE + ["f1_rank"]:
        m_, lo_, hi_, n_ = boot_excess(a, bs, SEED0 + hash(a) % 997, const=0.5)
        ver = equiv(lo_, hi_, PROBE_MARGIN)
        tag = "k커버리지" if a in KPROBE else "함수형"
        if a in KPROBE:
            okk.append(ver.startswith("✅"))
        DIFF[f"N2_{bs}_{a}"] = dict(mean=m_, lo=lo_, hi=hi_, n=n_)
        note = ""
        if a == "f1_rank" and bs == "rank":
            note = "   ← **기저 안**이라 정의상 0.5(검증 아님)"
        run.log(f"    {a:<9} [{tag}] {m_:+.4f} [{lo_:+.4f}, {hi_:+.4f}]  {ver}{note}")
    if bs == PRIMARY_BASIS:
        g_("N2a", "✅ 지지" if all(okk) else "⚠️ 미결",
           f"★ **전제** — k 프로브 {sum(okk)}/{len(KPROBE)} 등가 [{bs} 기저]")
        fm, flo, fhi, _ = boot_excess("f1_rank", bs, SEED0 + 7, const=0.5)
        g_("N2b", equiv(flo, fhi, PROBE_MARGIN),
           f"함수형(단조 변환) `f1_rank` {fm:+.4f} [{flo:+.4f}, {fhi:+.4f}]"
           f"   ← **실패는 봉인 사유가 아니라 캐비앳**. rank 기저를 병기한다")
SEALED = not VERD["N2a"].startswith("✅")
if SEALED:
    run.log("    ⛔ **기저가 `k` 계열을 다 덮지 못했다 — 아래 「수준」을 인용하지 않는다.**")
    run.log("       ★ 단 **차이**(N3a·N3b)와 기저별 반응은 공통 오염이 상쇄되므로 읽는다(R27 ②)")
if not VERD["N2b"].startswith("✅"):
    run.log("    ⚠️ **단조 비선형 성분은 안 지워졌다**(선형 기저의 한계). 수준을 인용할 때")
    run.log("       이 사실을 함께 적고, **rank 기저 결과를 나란히** 낸다")

# ── N3a · N3b ★ 폭 정합 등가성
for gname, (pa, pb) in zip(("N3a", "N3b"), WIDTH_PAIRS):
    w = SEGS[pa][1] - SEGS[pa][0]
    m_, lo_, hi_, n_ = boot_pair(pa, pb, PRIMARY_BASIS, SEED0 + 11, q=BONF3 * 100)
    DIFF[gname] = dict(mean=m_, lo=lo_, hi=hi_, n=n_, pair=[pa, pb], width=w)
    g_(gname, equiv(lo_, hi_, EQUIV_MARGIN),
       f"★ **폭 {w}** ({pa} 초과) − ({pb} 초과) **{m_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}] "
       f"· 여유 ±{EQUIV_MARGIN} · Bonf 3 · n={n_}")
    su, slo, shi, _ = boot_pair(pa, pb, PRIMARY_BASIS, SEED0 + 12)
    run.log(f"    (참고·미보정) {su:+.4f} [{slo:+.4f}, {shi:+.4f}] · "
            f"우월성 판정 {decide(slo, shi, 0.0, '>')}")
    for bs in ("rank", "quad", "hist"):      # ★ 함수형·기저 선택에 대한 삼각검증
        rm, rlo, rhi, _ = boot_pair(pa, pb, bs, SEED0 + 13)
        same = "✅ 같은 부호" if rm * m_ >= 0 else "⚠️ **부호 다름**"
        run.log(f"      [{bs:<4} 기저] {rm:+.4f} [{rlo:+.4f}, {rhi:+.4f}] "
                f"· {equiv(rlo, rhi, EQUIV_MARGIN)}  {same}")

# ── N4 ★ 이력 확장 기저에서 형태가 남는가 (rate hysteresis · post 는 안 넣는다)
m4, lo4, hi4, n4_ = boot_excess("stt", "hist", SEED0 + 13, q=BONF3 * 100)
DIFF["N4"] = dict(mean=m4, lo=lo4, hi=hi4, n=n4_)
g_("N4", decide(lo4, hi4, 0.0, ">"),
   f"★ **이력확장 기저** STT 초과 **{m4:+.4f}** [{lo4:+.4f}, {hi4:+.4f}] · Bonf 3 · n={n4_}"
   f"   ← 남으면 **rate hysteresis 로 설명 안 된다**")
run.log("    (기저별 수준·미보정)")
for a in ("stt", "stt_32", "stt_safe", "p_late", "p_full", "f3"):
    row = []
    for k in BASES:
        mm_, ll_, hh_, _ = boot_excess(a, k, SEED0 + 14)
        row.append(f"{k} {mm_:+.4f} [{ll_:+.4f},{hh_:+.4f}]")
    run.log(f"      {a:<9} " + " | ".join(row) + ("   ⛔ 봉인" if SEALED else ""))
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF; CONFIG["sealed"] = bool(SEALED)
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【N-E】 N5 층별 · N6 층화 삼각검증 (관문 아님)
run.log("\n" + "=" * 100)
run.log("【N-E】 N5 층별 · N6 삼각검증 · 강건성")
run.log("=" * 100)
ISOF = np.array([META[r]["iso_frac"] for r in RS])
LAY = (("고립 S", ISOF >= ISO_HI), ("혼합", (ISOF < ISO_HI) & (ISOF > ISO_LO)),
       ("런 우세", ISOF <= ISO_LO))
run.log(f"  N5 층별 — **잔차화는 {len(RS)}개체를 다 쓴다**(Q7-M 정합은 고립14·혼합3·런1 이었다)")
for nm_, msk in LAY:
    n_ = int(msk.sum())
    if n_ < 3:
        run.log(f"    {nm_:<8} {n_}개체 — 3 미만이라 CI 를 내지 않는다"); continue
    def sub(a, base="lin"):
        d = (RES[base][a] - NULL[base][a])[msk]
        d = d[np.isfinite(d)]
        rng = np.random.RandomState(SEED0 + 21)
        v = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(800)]
        return float(d.mean()), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))
    a1, l1, h1 = sub("stt"); a2, l2, h2 = sub("p_late"); a3, l3, h3 = sub("f3")
    run.log(f"    {nm_:<8} {n_:>2}개체 · STT {a1:+.4f} [{l1:+.4f},{h1:+.4f}] · "
            f"P_late {a2:+.4f} [{l2:+.4f},{h2:+.4f}] · f3 {a3:+.4f} [{l3:+.4f},{h3:+.4f}]")
    CONFIG.setdefault("layers", {})[nm_] = dict(n=n_, stt=a1, p_late=a2, f3=a3)

# ── N6 삼각검증 — 층화 판본 (f1 × f2_16 칸 · 개체 문턱 없음)
# ⚠️ 층화도 **칸 안에 두 클래스가 있어야** 한다 — 「비트 100%」가 아니다.
#    이득은 **개체 수준 문턱이 없다**는 것이다(정합은 15S·100쌍을 요구했다).
def strat_auc(sc, tt, key, min_s=1, min_n=1):
    uq, inv = np.unique(key, return_inverse=True)
    num = den = 0.0; ks_ = 0
    for j in range(len(uq)):
        m = inv == j
        s_, n_ = sc[m & tt], sc[m & ~tt]
        if len(s_) < min_s or len(n_) < min_n:
            continue
        ks_ += len(s_)
        gt = float((s_[:, None] > n_[None, :]).sum())
        eq = float((s_[:, None] == n_[None, :]).sum())
        num += gt + 0.5 * eq; den += float(len(s_) * len(n_))
    return (num / den if den >= 1 else float("nan")), ks_, den

run.log("\n  N6 삼각검증 — 층화(f1 × f2_16 칸) vs 잔차화")
STRAT = {a: np.full(len(RS), np.nan) for a in ("stt", "stt_32", "p_late", "p_full", "f3")}
SKEEP = np.zeros(len(RS), bool); SFRAC = np.full(len(RS), np.nan)
for i, r in enumerate(RS):
    F = FEAT[r]; tt = META[r]["tt"]
    pre_key = np.unique(np.round(F["f1"], 6), return_inverse=True)[1]
    f2b = np.floor(F[f"f2_{LB_K}"] / 0.02).astype(np.int64)
    key = pre_key.astype(np.int64) * (int(f2b.max() - f2b.min()) + 1) + (f2b - f2b.min())
    v0, ks0, den0 = strat_auc(score_of(r, "stt"), tt, key)
    if den0 < 1:
        continue
    SKEEP[i] = True; SFRAC[i] = ks0 / max(META[r]["pos"], 1)
    for a in STRAT:
        STRAT[a][i] = strat_auc(score_of(r, a), tt, key)[0]
run.log(f"    층화 가능 **{int(SKEEP.sum())}/{len(RS)}** 개체 · 남은 S 중앙 "
        f"{np.nanmedian(SFRAC[SKEEP]) if SKEEP.any() else float('nan'):.3f}"
        f"   (Q7-M 정합은 18개체 · 0.351 이었다)")
for a in STRAT:
    run.log(f"    {a:<9} 층화 {np.nanmean(STRAT[a][SKEEP]):.4f}  |  "
            f"선형잔차 {np.nanmean(RES['lin'][a]):.4f}  |  무잔차 {np.nanmean(RAW[a]):.4f}")
for pa, pb in WIDTH_PAIRS:
    ds = np.nanmean(STRAT[pa][SKEEP]) - np.nanmean(STRAT[pb][SKEEP])
    dr = DIFF[f"N3{'a' if (pa, pb) == WIDTH_PAIRS[0] else 'b'}"]["mean"]
    same = "✅ 같은 부호" if ds * dr >= 0 else "⚠️ **부호가 다르다**"
    run.log(f"    폭 {SEGS[pa][1]-SEGS[pa][0]} 차이 — 층화 {ds:+.4f} vs 잔차화 {dr:+.4f}  {same}")
CONFIG["strat"] = {a: float(np.nanmean(STRAT[a][SKEEP])) for a in STRAT}
CONFIG["strat_n"] = int(SKEEP.sum())
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【N-F】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다(네모 방지).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))
show = ["f1", "f2_16", "f3"] + PROBES + MORPH_ARMS
xs = np.arange(len(show))
ax[0].plot(xs, [np.nanmean(RAW[a]) for a in show], "o-", color="tab:gray", label="raw")
for k, c_ in (("lin", "tab:blue"), ("quad", "tab:green"), ("hist", "tab:red")):
    ax[0].plot(xs, [np.nanmean(RES[k][a]) for a in show], "o-", color=c_, label=k)
ax[0].axhline(0.5, ls="--", lw=0.8, color="k")
ax[0].set_xticks(xs); ax[0].set_xticklabels(show, rotation=60, fontsize=6, ha="right")
ax[0].set_ylabel("macro AUROC"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
ax[0].set_xlabel("arm  (probes are held out of the basis)")

ex = [np.nanmean(RES["lin"][a]) - np.nanmean(NULL["lin"][a]) for a in show]
ax[1].barh(xs, ex, color=["tab:green" if v > 0 else "tab:red" for v in ex])
ax[1].set_yticks(xs); ax[1].set_yticklabels(show, fontsize=6)
ax[1].axvline(0, color="k", lw=.8)
for m_ in (-PROBE_MARGIN, PROBE_MARGIN):
    ax[1].axvline(m_, ls=":", lw=.8, color="purple")
ax[1].set_xlabel("excess over own null (linear basis)"); ax[1].grid(alpha=.3, axis="x")

pairs = [f"w{SEGS[a][1]-SEGS[a][0]}: {a}-{b}" for a, b in WIDTH_PAIRS]
mm_ = [DIFF["N3a"]["mean"], DIFF["N3b"]["mean"]]
lo_ = [DIFF["N3a"]["lo"], DIFF["N3b"]["lo"]]
hi_ = [DIFF["N3a"]["hi"], DIFF["N3b"]["hi"]]
ax[2].errorbar(mm_, [0, 1], xerr=[np.array(mm_) - np.array(lo_),
                                  np.array(hi_) - np.array(mm_)],
               fmt="o", color="tab:blue", capsize=4)
ax[2].axvspan(-EQUIV_MARGIN, EQUIV_MARGIN, color="tab:green", alpha=.15)
ax[2].axvline(0, color="k", lw=.8)
ax[2].set_yticks([0, 1]); ax[2].set_yticklabels(pairs, fontsize=7)
ax[2].set_xlabel(f"width-matched difference  (shaded = equivalence +-{EQUIV_MARGIN})")
ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q7n_residualize", fig)
plt.close(fig)
display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("관문 요약")
run.log("=" * 100)
for k in ("N2a", "N2b", "N3a", "N3b", "N4"):
    run.log(f"  {k:<5}{VERD.get(k, '(미실행)')}")
run.log("")
ok_ = lambda k: VERD.get(k, "").startswith("✅")
un_ = lambda k: VERD.get(k, "").startswith("⛔")     # ★ 측정 불가는 어떤 분기도 안 탄다(R29 ②)
if SEALED:
    run.log("  ⛔ **N2a 미달 — 기저가 조기성의 `k` 계열을 다 덮지 못했다.**")
    run.log("     수준은 인용하지 않는다. **차이(N3a·N3b)만 읽는다**(R27 ②)")
else:
    run.log("  ★ **기저 밖 프로브가 전부 죽었다 — `k` 계열을 덮었다고 말할 수 있다.**")
for g_n in ("N3a", "N3b"):
    if un_(g_n):
        continue
    w = DIFF[g_n]["width"]; pa, pb = DIFF[g_n]["pair"]
    if ok_(g_n):
        run.log(f"  ★ **폭 {w} 정합 조건에서 `{pa}` 와 `{pb}` 는 등가다**"
                f"(CI ⊂ ±{EQUIV_MARGIN}) — **처음으로 「구별되지 않는다」를 확정**했다")
    else:
        run.log(f"  ⚠️ 폭 {w} — **미결**. 등가도 우월도 아니다. 「없다」로 쓰지 않는다(R29 ①)")
if un_("N4"):
    run.log("  ⛔ N4 측정 불가 — 결론으로 쓰지 않는다")
elif ok_("N4"):
    run.log("  ★ **이력을 확장해도 형태가 남는다 → rate hysteresis 로 설명되지 않는다.**")
    run.log("     다음은 **전이 가능성**(Q7-O 교차환자 템플릿)이다")
else:
    run.log("  ⚠️ N4 미결 — hysteresis 를 배제하지도 확인하지도 못했다")

run.finish({
    "exp_id": "quest46_q7n_residualize",
    "metric": "svdb_resid_stt_excess",
    "value": float(np.nanmean(RES["lin"]["stt"]) - np.nanmean(NULL["lin"]["stt"])),
    "passed": bool(not SEALED),
    "summary": ("정합 대신 잔차화로 조기성의 k 계열을 한꺼번에 제거하고, 폭을 맞춘 "
                "음성 대조와 등가성을 사전등록 여유로 판정했다."),
    "verdicts": VERD, "diffs": DIFF, "sealed": bool(SEALED),
    "raw": CONFIG.get("raw", {}), "residual": CONFIG.get("residual", {}),
    "null": CONFIG.get("null", {}), "layers": CONFIG.get("layers", {}),
    "strat": CONFIG.get("strat", {}), "rule_check": RULE_CHECK,
    "n_scored": len(RS), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-residualize`")